<a href="https://colab.research.google.com/github/ebi19912/AI/blob/main/VGG19_Autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

khanfashee_nih_chest_x_ray_14_224x224_resized_path = kagglehub.dataset_download('khanfashee/nih-chest-x-ray-14-224x224-resized')
rouhalahebrahimi_autoencoder_model_path = kagglehub.dataset_download('rouhalahebrahimi/autoencoder-model')

print('Data source import complete.')


In [ ]:

import os
import cv2
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ConvNeXtLarge
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Conv2D, BatchNormalization
from tensorflow.keras.layers import AvgPool2D, MaxPool2D, ReLU, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import backend as K
from tensorflow.keras.metrics import AUC

from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, confusion_matrix,
    matthews_corrcoef, average_precision_score, precision_recall_curve, roc_curve, roc_auc_score
)

from tensorflow.compat.v1.logging import INFO, set_verbosity
from tensorflow.python.framework.ops import disable_eager_execution
from matplotlib import rcParams
from keras.preprocessing import image
import os
import cv2
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ConvNeXtLarge
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Conv2D, BatchNormalization
from tensorflow.keras.layers import AvgPool2D, MaxPool2D, ReLU, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import backend as K
from tensorflow.keras.metrics import AUC

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, confusion_matrix,
    matthews_corrcoef, average_precision_score, precision_recall_curve, roc_curve, roc_auc_score
)

from tensorflow.compat.v1.logging import INFO, set_verbosity
from tensorflow.python.framework.ops import disable_eager_execution
from matplotlib import rcParams
from keras.preprocessing import image

In [ ]:
img_dir = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"


In [ ]:
dataframe = pd.read_csv("/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv")

columns = ["Image"]
for i in dataframe["Finding Labels"].values:
    for j in i.split("|"):
        if j not in columns:
            columns.append(j)
labels = columns.copy()
labels.remove("Image")

all_data = pd.DataFrame(columns=columns)
image_indices = dataframe["Image Index"].values
finding_labels = dataframe["Finding Labels"].values

data_list = []
for i in range(len(image_indices)):
    col = {c: 0 for c in columns}
    col["Image"] = image_indices[i]
    for j in finding_labels[i].split("|"):
        if j in col:
            col[j] = 1
    data_list.append(col)

all_data = pd.DataFrame(data_list)


train_df, test_val_df = train_test_split(all_data, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(test_val_df, test_size=0.5, random_state=42)

print(f"train num: {len(train_df)}")
print(f"validation num: {len(val_df)}")
print(f"test num: {len(test_df)}")

trainset = train_df.reset_index(drop=True)
valset = val_df.reset_index(drop=True)
testset = test_df.reset_index(drop=True)

In [ ]:
def isOverlap(s1, s2):
    total = set(s1).intersection(set(s2))
    return [len(total), total]

def overlapcheck(trainset, valset, testset):
    patid_train = []
    patid_val = []
    patid_test = []
    for name in trainset['Image'].values:
        patid_train.append(int(name.split("_")[0]))

    for name in valset['Image'].values:
        patid_val.append(int(name.split("_")[0]))

    for name in testset['Image'].values:
        patid_test.append(int(name.split("_")[0]))
    trte = isOverlap(patid_train, patid_test)
    teva = isOverlap(patid_test, patid_val)
    trva = isOverlap(patid_train, patid_val)
    print("Patient Overlap - Train and Test: ", trte[0])
    print("Patient Overlap - Test and Validation: ", teva[0])
    print("Patient Overlap - Train and Validation: ", trva[0])
    return trte, teva, trva

#Checking for overlaps between trainset, testset and validation set
trte, teva, trva = overlapcheck(trainset, valset, testset)

#Removing overlapping patients
for i in trva[1]:
    for name in trainset['Image'].values:
        if(int(name.split("_")[0]) == i):
            trainset.drop(trainset.loc[trainset['Image'] == name].index, inplace=True)

#Checking for overlaps after removing common patients
trte, teva, trva = overlapcheck(trainset, valset, testset)

In [ ]:
def label_counts(df):
    label_counts = df[labels].sum().sort_values(ascending=False)
    return label_counts

print("Training Data Label Counts:")
print(label_counts(trainset))
print("\nValidation Data Label Counts:")
print(label_counts(valset))
print("\nTest Data Label Counts:")
print(label_counts(testset))

In [ ]:
num = np.random.randint(trainset.shape[0])
sample = plt.imread(os.path.join(img_dir,trainset.iloc[[num]]["Image"].values[0]))
plt.figure(figsize=(15, 15))
plt.title(dataframe[dataframe["Image Index"] == trainset.iloc[[num]]["Image"].values[0]].values[0][1])
plt.imshow(sample, cmap = 'gray')
plt.colorbar()
trainset.iloc[[num]]

print("Maximum Pixel Value: ", sample.max())
print("Minimum Pixel Value: ", sample.min())
print(f"Image dimension: {sample.shape[0]} x {sample.shape[1]} ")

fig, ax = plt.subplots(figsize=(25, 10))
plt.xlabel("Pixel Values")
print("Mean - Pixel Value: ", sample.mean())
print("Std Deviation Pixel Value: ", sample.std())
sns.histplot(sample.ravel(), ax = ax, kde = True)

In [ ]:
def adjust_contrast_and_saturation(image):
  # Generate a random contrast factor between 0.4 and 0.9
  contrast_factor = tf.random.uniform([], 0.4, 0.9)
  image = tf.image.adjust_contrast(image, contrast_factor)

  # Generate a random saturation factor between 0.4 and 0.9
  saturation_factor = tf.random.uniform([], 0.4, 0.9)
  image = tf.image.adjust_saturation(image, saturation_factor)

  return image

traingen = ImageDataGenerator(
    featurewise_center=True,
    featurewise_std_normalization=True,
    brightness_range=[0.3, 1.2],
    zoom_range=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    preprocessing_function=adjust_contrast_and_saturation
)


sample_generator = ImageDataGenerator().flow_from_dataframe(
    dataframe=trainset,
    directory=img_dir,
    x_col="Image",
    y_col=labels,
    class_mode="raw",
    batch_size=10000, # یک بچ بزرگ برای محاسبه آمار دقیق‌تر
    shuffle=True,
    target_size=(224, 224)
)
sample_images, _ = next(sample_generator)
traingen.fit(sample_images)


val_test_gen = ImageDataGenerator(
    featurewise_center=True,
    featurewise_std_normalization=True,
)

val_test_gen.mean = traingen.mean
val_test_gen.std = traingen.std


bach = 8
traingenerator = traingen.flow_from_dataframe(
        dataframe=trainset,
        directory=img_dir,
        x_col="Image",
        y_col= labels,
        class_mode="raw",
        batch_size= bach,
        shuffle=True,
        target_size=(224,224)
)

valgenerator = val_test_gen.flow_from_dataframe(
    dataframe=valset,
    directory=img_dir,
    x_col="Image",
    y_col=labels,
    class_mode="raw",
    batch_size=bach,
    shuffle=False,
    target_size=(224, 224)
)

testgenerator = val_test_gen.flow_from_dataframe(
    dataframe=testset,
    directory=img_dir,
    x_col="Image",
    y_col=labels,
    class_mode="raw",
    batch_size=bach,
    shuffle=False,
    target_size=(224, 224)
)


num = np.random.randint(len(traingenerator))
item, value = traingenerator.__getitem__(num)
plt.figure(figsize=(15, 15))
plt.imshow(item[0], cmap='gray')
plt.colorbar()
plt.show()

In [ ]:
# Calculate positive and negative frequencies for each label
positive_freqs = trainset[labels].sum().values / trainset.shape[0]  # Assuming labels are 1 for positive, 0 for negative
negative_freqs = 1 - positive_freqs

data = {
    'Class': labels,
    'Positive': positive_freqs, #* negative_freqs, #Removed
    'Negative': negative_freqs #* positive_freqs #Removed
}

X_axis = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(X_axis-0.2, data['Positive'], width=0.4, color='b', label = "Positive")
ax.bar(X_axis+0.2, data['Negative'], width=0.4, color='r', label = 'Negative')
plt.xticks(X_axis, labels, rotation = 90)
plt.legend()
plt.figure(figsize=(20,15))

In [ ]:
# Assuming you have your labels and save directory defined

save_dir = '/kaggle/working/'
os.makedirs(save_dir, exist_ok=True)

# Define a function to create the model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, concatenate, Flatten, Lambda
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.applications import VGG19
import tensorflow as tf
import os
def create_model():
    # Load pretrained encoder
    encoder = load_model('/kaggle/input/autoencoder-model/encoder_model.keras', compile=False)
    for layer in encoder.layers:
        layer.trainable = False

    # Input
    input_tensor = Input(shape=(224, 224, 3), name='input_rgb')

    # Encoder branch (grayscale + resize to match encoder input)
    encoder_input = Lambda(lambda x: tf.image.resize(tf.reduce_mean(x, axis=-1, keepdims=True), (128, 128)),
                           name='rgb_to_gray_resize')(input_tensor)
    encoder_output = encoder(encoder_input)
    encoder_flat = Flatten(name='encoder_flat')(encoder_output)

    # VGG19 branch
    vgg_base = VGG19(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    for layer in vgg_base.layers[:-5]:
        layer.trainable = False

    vgg_feat = vgg_base(input_tensor)
    vgg_gap = GlobalAveragePooling2D(name='vgg_gap')(vgg_feat)
    vgg_dense = Dense(256, activation='relu', name='vgg_dense')(vgg_gap)

    # Merge both features
    merged = concatenate([encoder_flat, vgg_dense], name='concat_features')

    # Dense classifier
    x = Dense(512, activation='relu', name='dense1')(merged)
    x = Dense(256, activation='relu', name='dense2')(x)
    output = Dense(len(labels), activation='sigmoid', name='output')(x)

    model = Model(inputs=input_tensor, outputs=output)
    return model



model = create_model()
model.summary()

In [ ]:
def weighted_cross_entropy(y_true, y_pred):
    # Calculate positive and negative weights based on class frequencies in your dataset.
    positive_weights = tf.constant(positive_freqs, dtype=tf.float32)  # Example positive weights
    negative_weights = tf.constant(negative_freqs, dtype=tf.float32)  # Example negative weights

    # Calculate the weighted loss
    loss = -(y_true * tf.math.log(y_pred + 1e-7) * positive_weights + (1 - y_true) * tf.math.log(1 - y_pred + 1e-7) * negative_weights)

    return tf.reduce_mean(loss)

In [ ]:
# Function to train the model and save it
def train_and_save_model(model_name):
    # Compile the model
    model.compile(optimizer=Adam(learning_rate=3e-5), loss=weighted_cross_entropy, metrics=['accuracy', AUC(curve='ROC', name='auc')])

    # Define callbacks
    checkpoint = ModelCheckpoint(f'{model_name}.keras',
                                 monitor='val_auc',
                                 verbose=1,
                                 save_best_only=True,
                                 mode='max')
    early_stopping = EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True)

    # Train the model
    history = model.fit(
        traingenerator,
        epochs=100,
        validation_data=valgenerator,
        callbacks=[checkpoint, early_stopping]
    )

    return history

# Train the model and save it
history = train_and_save_model('VGG19_Autoencoder_chest_xray_ensemble')

In [ ]:
predicted_vals = model.predict(testgenerator)

In [ ]:
# Initialize lists to store evaluation metrics for each label
precision_list = []
recall_list = []
f1_list = []
accuracy_list = []
mcc_list = []
auc_pr_list = []
best_thresholds = []
# Iterate through labels and calculate metrics
for i in range(len(labels)):
    try:
        gt = np.array(testgenerator.labels[:, i])
        pred_probs = predicted_vals[:, i]

        # Find optimal threshold based on F1-score
        best_f1 = -1
        best_thresh = 0
        for thresh in np.arange(0.0001, 0.0999999, 0.0001):  # Iterate through different thresholds
            pred = (pred_probs > thresh).astype(int)
            f1 = f1_score(gt, pred)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = thresh

        best_thresholds.append(best_thresh)
        print(f"Best threshold for {labels[i]}: {best_thresh}")

        # Use the best threshold for evaluation
        pred = (pred_probs > best_thresh).astype(int)

        precision = precision_score(gt, pred)
        recall = recall_score(gt, pred)
        f1 = f1_score(gt, pred)
        accuracy = accuracy_score(gt, pred)
        mcc = matthews_corrcoef(gt, pred)
        auc_pr = average_precision_score(gt, pred_probs)

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
        accuracy_list.append(accuracy)
        mcc_list.append(mcc)
        auc_pr_list.append(auc_pr)

        # Calculate Confusion Matrix
        cm = confusion_matrix(gt, pred)
        print(f"Confusion Matrix for {labels[i]} (Threshold: {best_thresh}):")
        print(cm)
        # Calculate Specificity
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp)
        print(f"Specificity for {labels[i]}: {specificity}")

    except ValueError:
        pass


# Store the metrics in a DataFrame for better presentation
metrics_df = pd.DataFrame({
    'Label': labels,
    'Best Threshold': best_thresholds,
    'Precision': precision_list,
    'Recall': recall_list,
    'F1-Score': f1_list,
    'Accuracy': accuracy_list,
    'MCC': mcc_list,
    'AUC-PR': auc_pr_list
})

print(metrics_df)

# Plot ROC and PR curves for each label in one plot
plt.figure(figsize=(12, 6))

# ROC curve
plt.subplot(1, 2, 1)
for i in range(len(labels)):
    try:
        gt = np.array(testgenerator.labels[:, i])
        pred = predicted_vals[:, i]
        fpr, tpr, thresholds = roc_curve(gt, pred)

        # Calculate Youden's J for each threshold
        youden_j = tpr - fpr

        # Find the optimal threshold based on Youden's J
        optimal_idx = np.argmax(youden_j)
        optimal_threshold = thresholds[optimal_idx]

        roc_auc = roc_auc_score(gt, pred)
        plt.plot(fpr, tpr, label=f"{labels[i]} (AUC = {roc_auc:.2f}, Optimal Threshold = {optimal_threshold:.2f})")
    except ValueError:
        pass
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')

# PR curve
plt.subplot(1, 2, 2)
for i in range(len(labels)):
    try:
        gt = np.array(testgenerator.labels[:, i])
        pred = predicted_vals[:, i]
        precision, recall, _ = precision_recall_curve(gt, pred)
        average_precision = average_precision_score(gt, pred)
        plt.plot(recall, precision, label=f"{labels[i]} (AP = {average_precision:.2f})")
    except ValueError:
        pass
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             matthews_corrcoef, average_precision_score, confusion_matrix,
                             roc_curve, roc_auc_score, precision_recall_curve)

weighted_precision_list = []
weighted_recall_list = []
weighted_f1_list = []
accuracy_list = []
mcc_list = []
auc_pr_list = []
best_thresholds = []

all_gt = np.array(testgenerator.labels)
all_pred_probs = predicted_vals

for i in range(len(labels)):
    try:
        gt = all_gt[:, i]
        pred_probs = all_pred_probs[:, i]

        best_f1 = -1
        best_thresh = 0
        for thresh in np.arange(0.0001, 0.0999999, 0.0001):
            pred = (pred_probs > thresh).astype(int)
            f1 = f1_score(gt, pred, average="weighted")
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = thresh

        best_thresholds.append(best_thresh)
        print(f"Best threshold for {labels[i]}: {best_thresh}")

        pred = (pred_probs > best_thresh).astype(int)

        weighted_precision = precision_score(gt, pred, average="weighted")
        weighted_recall = recall_score(gt, pred, average="weighted")
        weighted_f1 = f1_score(gt, pred, average="weighted")
        accuracy = accuracy_score(gt, pred)
        mcc = matthews_corrcoef(gt, pred)
        auc_pr = average_precision_score(gt, pred_probs)

        weighted_precision_list.append(weighted_precision)
        weighted_recall_list.append(weighted_recall)
        weighted_f1_list.append(weighted_f1)
        accuracy_list.append(accuracy)
        mcc_list.append(mcc)
        auc_pr_list.append(auc_pr)

        cm = confusion_matrix(gt, pred)
        print(f"Confusion Matrix for {labels[i]} (Threshold: {best_thresh}):")
        print(cm)

        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            print(f"Specificity for {labels[i]}: {specificity}")

    except ValueError:
        pass

metrics_df = pd.DataFrame({
    'Label': labels,
    'Best Threshold': best_thresholds,
    'Weighted Precision': weighted_precision_list,
    'Weighted Recall': weighted_recall_list,
    'Weighted F1-Score': weighted_f1_list,
    'Accuracy': accuracy_list,
    'MCC': mcc_list,
    'AUC-PR': auc_pr_list
})

print(metrics_df)

print("\nOverall Weighted Metrics:")
print(f"Mean Weighted Precision: {np.mean(weighted_precision_list):.4f}")
print(f"Mean Weighted Recall: {np.mean(weighted_recall_list):.4f}")
print(f"Mean Weighted F1-Score: {np.mean(weighted_f1_list):.4f}")
print(f"Mean Accuracy: {np.mean(accuracy_list):.4f}")
print(f"Mean MCC: {np.mean(mcc_list):.4f}")
print(f"Mean AUC-PR: {np.mean(auc_pr_list):.4f}")

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
for i in range(len(labels)):
    try:
        gt = all_gt[:, i]
        pred = all_pred_probs[:, i]
        fpr, tpr, thresholds = roc_curve(gt, pred)

        youden_j = tpr - fpr
        optimal_idx = np.argmax(youden_j)
        optimal_threshold = thresholds[optimal_idx]

        roc_auc = roc_auc_score(gt, pred)
        plt.plot(fpr, tpr, label=f"{labels[i]} (AUC = {roc_auc:.2f}, Optimal Threshold = {optimal_threshold:.2f})")
    except ValueError:
        pass
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')

plt.subplot(1, 2, 2)
for i in range(len(labels)):
    try:
        gt = all_gt[:, i]
        pred = all_pred_probs[:, i]
        precision, recall, _ = precision_recall_curve(gt, pred)
        average_precision = average_precision_score(gt, pred)
        plt.plot(recall, precision, label=f"{labels[i]} (AP = {average_precision:.2f})")
    except ValueError:
        pass
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')

plt.tight_layout()
plt.show()


In [ ]:
# Initialize lists to store evaluation metrics for each label
precision_list = []
recall_list = []
f1_list = []
accuracy_list = []
mcc_list = []
auc_pr_list = []
auc_roc_list = []
best_thresholds = []


for i in range(len(labels)):
    try:
        gt = np.array(testgenerator.labels[:, i])
        pred_probs = predicted_vals[:, i]

        # --- Find optimal threshold based on Youden's J from ROC ---
        fpr, tpr, thresholds = roc_curve(gt, pred_probs)
        youden_j = tpr - fpr
        optimal_idx = np.argmax(youden_j)
        best_thresh = thresholds[optimal_idx]
        best_thresholds.append(best_thresh)

        # Calculate AUC metrics
        auc_roc = roc_auc_score(gt, pred_probs)
        auc_pr = average_precision_score(gt, pred_probs)

        # Predictions using best threshold
        pred = (pred_probs > best_thresh).astype(int)

        # Other metrics
        precision = precision_score(gt, pred)
        recall = recall_score(gt, pred)
        f1 = f1_score(gt, pred)
        accuracy = accuracy_score(gt, pred)
        mcc = matthews_corrcoef(gt, pred)

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
        accuracy_list.append(accuracy)
        mcc_list.append(mcc)
        auc_pr_list.append(auc_pr)
        auc_roc_list.append(auc_roc)

        # Confusion Matrix + Specificity
        cm = confusion_matrix(gt, pred)
        print(f"\nLabel: {labels[i]}")
        print(f"Best threshold (Youden's J): {best_thresh}")
        print(f"AUC-ROC: {auc_roc:.4f}, AUC-PR: {auc_pr:.4f}")
        print("Confusion Matrix:")
        print(cm)
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp)
        print(f"Specificity: {specificity:.4f}")

    except ValueError:
        pass

# Store metrics in DataFrame
metrics_df = pd.DataFrame({
    'Label': labels,
    'Best Threshold (YoudenJ)': best_thresholds,
    'Precision': precision_list,
    'Recall': recall_list,
    'F1-Score': f1_list,
    'Accuracy': accuracy_list,
    'MCC': mcc_list,
    'AUC-ROC': auc_roc_list,
    'AUC-PR': auc_pr_list
})

print("\nFinal Metrics Table:")
print(metrics_df)

# ---- Plot ROC and PR Curves ----
plt.figure(figsize=(12, 6))

# ROC curve
plt.subplot(1, 2, 1)
for i in range(len(labels)):
    try:
        gt = np.array(testgenerator.labels[:, i])
        pred = predicted_vals[:, i]
        fpr, tpr, thresholds = roc_curve(gt, pred)
        roc_auc = roc_auc_score(gt, pred)
        plt.plot(fpr, tpr, label=f"{labels[i]} (AUC = {roc_auc:.2f})")
    except ValueError:
        pass
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')

# PR curve
plt.subplot(1, 2, 2)
for i in range(len(labels)):
    try:
        gt = np.array(testgenerator.labels[:, i])
        pred = predicted_vals[:, i]
        precision, recall, _ = precision_recall_curve(gt, pred)
        average_precision = average_precision_score(gt, pred)
        plt.plot(recall, precision, label=f"{labels[i]} (AP = {average_precision:.2f})")
    except ValueError:
        pass
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')

plt.tight_layout()
plt.show()
